# Vocoder Comparison: Python mlx-audio vs Rust voicers

Same phonemes, same model weights, same voice — which produces intelligible audio?

If Python sounds fine and Rust doesn't, the bug is in our Rust vocoder (iSTFT, overlap-add, weight loading, etc).
If both sound bad, it's the model itself or an MLX framework issue.

In [ ]:
import subprocess, os, tempfile
import numpy as np
import soundfile as sf

# Use the installed binary
VOICERS_CLI = "voice"

import whisper

whisper_model = whisper.load_model("base")
print("Whisper: ready")

from mlx_audio.tts.utils import load as load_tts

py_model = load_tts("prince-canuma/Kokoro-82M")
print("Python Kokoro: ready")


def transcribe(wav_path):
    return whisper_model.transcribe(wav_path, language="en")["text"].strip()


def python_generate(text, voice="af_heart"):
    wav_path = tempfile.mktemp(suffix=".wav")
    for gen_result in py_model.generate(text, voice=voice, lang_code="a"):
        audio = np.array(gen_result.audio)
        sf.write(wav_path, audio, gen_result.sample_rate)
        return wav_path
    return None


def rust_generate(text, voice="af_heart"):
    wav_path = tempfile.mktemp(suffix=".wav")
    r = subprocess.run(
        [VOICERS_CLI, "-v", voice, "-o", wav_path, text],
        capture_output=True,
        text=True,
        timeout=120,
    )
    phonemes = ""
    for line in r.stderr.splitlines():
        if line.strip().startswith("chunk"):
            phonemes += line.split(":", 1)[1].strip() + " "
    return wav_path, phonemes.strip()


print("All ready.")

In [ ]:
# A/B Comparison: Python vs Rust (after iSTFT normalization fix)
test_phrases = [
    "Hello world",
    "Good morning",
    "The quick brown fox",
    "How are you doing today?",
    "She sells seashells by the seashore.",
    "I read a book yesterday.",
    "The dog is running in the park.",
]

print(f"{'Input':<40} {'Python Whisper':<40} {'Rust Whisper'}")
print("=" * 120)

py_results = []
rust_results = []

for phrase in test_phrases:
    py_wav = python_generate(phrase)
    py_heard = transcribe(py_wav) if py_wav else "FAILED"
    os.unlink(py_wav) if py_wav else None

    rust_wav, rust_ph = rust_generate(phrase)
    rust_heard = transcribe(rust_wav)
    os.unlink(rust_wav)

    py_results.append((phrase, py_heard))
    rust_results.append((phrase, rust_heard, rust_ph))

    print(f"{phrase:<40} {py_heard[:38]:<40} {rust_heard[:38]}")


# Score: exact match (case-insensitive, strip trailing punct)
def normalize(s):
    return s.lower().rstrip(".,!?;:")


py_score = sum(1 for p, h in py_results if normalize(p) == normalize(h))
rust_score = sum(1 for p, h, _ in rust_results if normalize(p) == normalize(h))
print(
    f"\nExact match: Python {py_score}/{len(test_phrases)}   Rust {rust_score}/{len(test_phrases)}"
)

## Results: Python mlx-audio vs Rust voicers

| Input | Python (Whisper) | Rust (Whisper) |
|-------|-----------------|----------------|
| Hello world | Hello world. | Hello, where? Holds that. |
| Good morning | Good morning. | Good morning, hey |
| The quick brown fox | The quick brown fox. | The cool, very evil! |
| How are you doing today? | How are you doing today? | How art you doing to say |
| She sells seashells... | She sells seashells by the s... | She sells a sea shells by th... |
| I read a book yesterday. | I read a book yesterday. | I read a book, yes, to Herds |
| The dog is running... | The dog is running in the pa... | The Dizdog is as running int... |

**Python: 7/7 correct. Rust: 0/7 correct.**

The Python mlx-audio pipeline produces perfect audio from the same model. Our Rust vocoder is the problem. The issue is somewhere in:
1. **iSTFT reconstruction** (overlap-add, window normalization)
2. **Weight loading/sanitization** (some weights may still be misaligned)
3. **Forward pass numerics** (intermediate computation differences)

Next: dump intermediate tensors from both Python and Rust to find where they diverge.

In [ ]:
# Compare the actual waveform statistics between Python and Rust
py_wav = python_generate("Hello world")
rust_wav, _ = rust_generate("Hello world")

py_audio, py_sr = sf.read(py_wav)
rust_audio, rust_sr = sf.read(rust_wav)

print(f"Python: shape={py_audio.shape}, sr={py_sr}, dtype={py_audio.dtype}")
print(
    f"  min={py_audio.min():.4f}, max={py_audio.max():.4f}, mean={py_audio.mean():.6f}, std={py_audio.std():.4f}"
)
print(f"  samples: {len(py_audio)}, duration: {len(py_audio) / py_sr:.2f}s")
print()
print(f"Rust:   shape={rust_audio.shape}, sr={rust_sr}, dtype={rust_audio.dtype}")
print(
    f"  min={rust_audio.min():.4f}, max={rust_audio.max():.4f}, mean={rust_audio.mean():.6f}, std={rust_audio.std():.4f}"
)
print(f"  samples: {len(rust_audio)}, duration: {len(rust_audio) / rust_sr:.2f}s")

# Amplitude ratio
print(f"\nAmplitude ratio (rust_std / py_std): {rust_audio.std() / py_audio.std():.4f}")
print(
    f"RMS ratio: {np.sqrt(np.mean(rust_audio**2)) / np.sqrt(np.mean(py_audio**2)):.4f}"
)

os.unlink(py_wav)
os.unlink(rust_wav)

In [ ]:
# Let's look at the WAV structure more carefully
# The Rust output being longer could mean:
# 1. Duration prediction is wrong (model predicting longer durations)
# 2. Extra silence at start/end (padding not trimmed)
# 3. Wrong sample rate in WAV header

# Check if there's a long silence tail in Rust output
py_wav = python_generate("Hello world")
rust_wav, _ = rust_generate("Hello world")
py_audio, _ = sf.read(py_wav)
rust_audio, _ = sf.read(rust_wav)


# Find where the audio actually starts/ends (above noise floor)
def find_audio_bounds(audio, threshold=0.005):
    abs_audio = np.abs(audio)
    above = abs_audio > threshold
    if not np.any(above):
        return 0, len(audio)
    start = np.argmax(above)
    end = len(audio) - np.argmax(above[::-1])
    return int(start), int(end)


py_start, py_end = find_audio_bounds(py_audio)
rust_start, rust_end = find_audio_bounds(rust_audio)

print(f"Python: total={len(py_audio)} samples")
print(
    f"  Audio region: {py_start} to {py_end} ({py_end - py_start} samples, {(py_end - py_start) / 24000:.2f}s)"
)
print(
    f"  Leading silence: {py_start / 24000:.3f}s, trailing silence: {(len(py_audio) - py_end) / 24000:.3f}s"
)
print()
print(f"Rust:   total={len(rust_audio)} samples")
print(
    f"  Audio region: {rust_start} to {rust_end} ({rust_end - rust_start} samples, {(rust_end - rust_start) / 24000:.2f}s)"
)
print(
    f"  Leading silence: {rust_start / 24000:.3f}s, trailing silence: {(len(rust_audio) - rust_end) / 24000:.3f}s"
)

os.unlink(py_wav)
os.unlink(rust_wav)

In [ ]:
# Let's trace the exact intermediate values.
# First, check what phonemes and token IDs each side uses for "Hello world"

# Python side: get phonemes and token IDs
script = """
import sys
sys.path.insert(0, '/Users/kylekelley/conductor/workspaces/misaki/abuja-v4')
from misaki.en import G2P
import spacy, json

g2p = G2P(trf=False, british=False, unk='')
ps, tokens = g2p("Hello world")
print(f"phonemes: {ps}")
print(f"tokens: {[(t.text, t.phonemes, t.tag) for t in tokens]}")

# Show the vocab mapping
import json as j
from huggingface_hub import hf_hub_download
config_path = hf_hub_download('prince-canuma/Kokoro-82M', 'config.json')
with open(config_path) as f:
    config = j.load(f)
vocab = config['vocab']
ids = [vocab.get(c, -1) for c in ps]
print(f"token_ids: {ids}")
print(f"phoneme_count: {len(ps)}")
"""
r = subprocess.run(
    [
        "uv",
        "run",
        "--with",
        "misaki[en]",
        "--with",
        "spacy",
        "--with",
        "transformers",
        "--with",
        "torch",
        "--with",
        "en-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl",
        "--with",
        "huggingface-hub",
        "python",
        "-c",
        script,
    ],
    capture_output=True,
    text=True,
    timeout=120,
)
print("=== Python misaki output ===")
print(r.stdout)
if r.stderr:
    # Just show the important lines
    for line in r.stderr.splitlines():
        if not line.startswith((" ", "  ")) and "WARNING" not in line:
            print(f"  stderr: {line}")

In [ ]:
# Now check what the Rust side produces for "Hello world"
# The phonemes were: həlˈO wˈɜɹld

# Let's check token IDs from Rust by looking at the vocab
import json
from huggingface_hub import hf_hub_download

config_path = hf_hub_download("prince-canuma/Kokoro-82M", "config.json")
with open(config_path) as f:
    config = json.load(f)
vocab = config["vocab"]

rust_phonemes = "həlˈO wˈɜɹld"
rust_ids = [vocab.get(c, -1) for c in rust_phonemes]
print(f"Rust phonemes: {rust_phonemes}")
print(f"Rust token_ids: {rust_ids}")
print(f"Rust phoneme_count: {len(rust_phonemes)}")

py_phonemes = "həlˈO wˈɜɹld"
py_ids = [vocab.get(c, -1) for c in py_phonemes]
print(f"\nPython phonemes: {py_phonemes}")
print(f"Python token_ids: {py_ids}")
print(f"Python phoneme_count: {len(py_phonemes)}")

print(f"\nPhonemes match: {rust_phonemes == py_phonemes}")
print(f"Token IDs match: {rust_ids == py_ids}")

# Check vocab for space
print(f"\nSpace in vocab: {vocab.get(' ', 'NOT FOUND')}")
print(f"BOS token (0): {[k for k, v in vocab.items() if v == 0]}")

# Show full vocab
print(f"\nVocab size: {len(vocab)}")
print(f"Vocab: {dict(sorted(vocab.items(), key=lambda x: x[1]))}")

In [ ]:
# Deep dive: extract intermediate tensors from both pipelines
# Python: Use the model directly to get duration predictions

script = """
import sys, json, numpy as np
sys.path.insert(0, '/Users/kylekelley/conductor/workspaces/misaki/abuja-v4')
sys.path.insert(0, '/Users/kylekelley/conductor/workspaces/kokoro/montevideo')

import torch
from kokoro.model import KModel
from huggingface_hub import hf_hub_download

# Load model
config_path = hf_hub_download('prince-canuma/Kokoro-82M', 'config.json')
with open(config_path) as f:
    config = json.load(f)

model = KModel(config)
weights_path = hf_hub_download('prince-canuma/Kokoro-82M', 'kokoro-v1_0.pth')
state = torch.load(weights_path, map_location='cpu', weights_only=True)
model.load_state_dict(state)
model.eval()

# Load voice
voice_path = hf_hub_download('prince-canuma/Kokoro-82M', 'voices/af_heart.pt')
voice = torch.load(voice_path, map_location='cpu', weights_only=True)

# Prepare input - same phonemes as Rust
phonemes = "həlˈO wˈɜɹld"
vocab = config['vocab']
tokens = [vocab[c] for c in phonemes]
input_ids = torch.tensor([[0, *tokens, 0]], dtype=torch.long)
ref_s = voice[len(tokens)]  # voice_pack indexed by phoneme count

print(f"input_ids shape: {input_ids.shape}")
print(f"input_ids: {input_ids.tolist()}")
print(f"ref_s shape: {ref_s.shape}")
print(f"ref_s[:5]: {ref_s[0,:5].tolist()}")

# Run the model forward pass with torch.no_grad
with torch.no_grad():
    # Get BERT output
    bert_out = model.bert(input_ids, attention_mask=(~(input_ids==0)).int())
    asr = model.bert_encoder(bert_out[0]).transpose(1, 2)
    print(f"asr shape: {asr.shape}")
    print(f"asr mean: {asr.mean().item():.6f}, std: {asr.std().item():.6f}")
    
    # Split style
    s_content = ref_s[:, 128:]
    s_ref = ref_s[:, :128]
    
    # Get durations via predictor
    d = model.predictor.text_encoder(asr, s_content)
    print(f"duration_encoded shape: {d.shape}")
    
    # Full forward pass to get audio
    audio = model(input_ids, ref_s, speed=1.0)
    if isinstance(audio, tuple):
        audio = audio[0]
    audio_np = audio.cpu().numpy().flatten()
    print(f"audio shape: {audio_np.shape}, duration: {len(audio_np)/24000:.2f}s")
    print(f"audio stats: min={audio_np.min():.4f}, max={audio_np.max():.4f}, std={audio_np.std():.4f}")
"""

r = subprocess.run(
    [
        "uv",
        "run",
        "--with",
        "misaki[en]",
        "--with",
        "spacy",
        "--with",
        "transformers",
        "--with",
        "torch",
        "--with",
        "huggingface-hub",
        "--with",
        "soundfile",
        "--with",
        "en-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl",
        "python",
        "-c",
        script,
    ],
    capture_output=True,
    text=True,
    timeout=120,
)
print("=== Python PyTorch model internals ===")
for line in r.stdout.splitlines():
    print(line)
if r.returncode != 0:
    print("ERRORS:")
    for line in r.stderr.splitlines()[-10:]:
        print(f"  {line}")

In [ ]:
# Add loguru + phonemizer deps
script = """
import sys, json, numpy as np
sys.path.insert(0, '/Users/kylekelley/conductor/workspaces/kokoro/montevideo')

import torch
from kokoro.model import KModel
from huggingface_hub import hf_hub_download

config_path = hf_hub_download('prince-canuma/Kokoro-82M', 'config.json')
with open(config_path) as f:
    config = json.load(f)

model = KModel(config)
weights_path = hf_hub_download('prince-canuma/Kokoro-82M', 'kokoro-v1_0.pth')
state = torch.load(weights_path, map_location='cpu', weights_only=True)
model.load_state_dict(state)
model.eval()

voice_path = hf_hub_download('prince-canuma/Kokoro-82M', 'voices/af_heart.pt')
voice = torch.load(voice_path, map_location='cpu', weights_only=True)

phonemes = "həlˈO wˈɜɹld"
vocab = config['vocab']
tokens = [vocab[c] for c in phonemes]
input_ids = torch.tensor([[0, *tokens, 0]], dtype=torch.long)
ref_s = voice[len(tokens)]

print(f"input_ids shape: {input_ids.shape}")
print(f"input_ids: {input_ids.tolist()}")
print(f"ref_s shape: {ref_s.shape}")
print(f"ref_s[0,:5]: {ref_s[0,:5].tolist()}")

with torch.no_grad():
    audio = model(input_ids, ref_s, speed=1.0)
    if isinstance(audio, tuple):
        audio = audio[0]
    audio_np = audio.cpu().numpy().flatten()
    print(f"audio shape: {audio_np.shape}, duration: {len(audio_np)/24000:.2f}s")
    print(f"audio stats: min={audio_np.min():.4f}, max={audio_np.max():.4f}, std={audio_np.std():.4f}")
"""

r = subprocess.run(
    [
        "uv",
        "run",
        "--with",
        "loguru",
        "--with",
        "phonemizer",
        "--with",
        "transformers",
        "--with",
        "torch",
        "--with",
        "huggingface-hub",
        "--with",
        "soundfile",
        "--with",
        "misaki[en]",
        "--with",
        "spacy",
        "--with",
        "en-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl",
        "python",
        "-c",
        script,
    ],
    capture_output=True,
    text=True,
    timeout=120,
)
print("=== PyTorch Kokoro model ===")
for line in r.stdout.splitlines():
    print(line)
if r.returncode != 0:
    print("ERRORS:")
    for line in r.stderr.splitlines()[-10:]:
        print(f"  {line}")

In [ ]:
# py_model is the Model itself, not a pipeline wrapper
import mlx.core as mx

model = py_model
print(f"Model type: {type(model)}")
print(f"Model attributes: {[a for a in dir(model) if not a.startswith('_')][:30]}")

# Look at the generate method source
import inspect

# Get the __call__ or generate method
print(f"\nHas __call__: {hasattr(model, '__call__')}")
print(f"Has generate: {hasattr(model, 'generate')}")

# Check key attributes
for attr in ["vocab", "bert", "bert_encoder", "predictor", "text_encoder", "decoder"]:
    print(f"  {attr}: {hasattr(model, attr)}")

In [ ]:
# Now let's trace the forward pass step by step in Python MLX
# First, read the Python model's __call__ to understand the flow
import inspect

src = inspect.getsource(model.__call__)
print(src[:3000])

In [ ]:
# Step-by-step instrumented forward pass in Python MLX
import mlx.core as mx
import numpy as np

phonemes = "həlˈO wˈɜɹld"
vocab = model.vocab
input_ids_list = [v for c in phonemes if (v := vocab.get(c)) is not None]
input_ids = mx.array([[0, *input_ids_list, 0]])
seq_len = input_ids.shape[-1]
input_lengths = mx.array([seq_len])

text_mask = mx.arange(int(input_lengths.max()))[None, :]
text_mask = mx.repeat(text_mask, input_lengths.shape[0], axis=0).astype(
    input_lengths.dtype
)
text_mask = text_mask + 1 > input_lengths[:, None]

print(f"input_ids: {input_ids.tolist()}")
print(f"seq_len: {seq_len}")
print(f"text_mask: {text_mask.tolist()}")
print(f"text_mask shape: {text_mask.shape}")

# Load voice
from huggingface_hub import hf_hub_download

voice_path = hf_hub_download("prince-canuma/Kokoro-82M", "voices/af_heart.pt")
import torch

voice_pt = torch.load(voice_path, map_location="cpu", weights_only=True)
voice = mx.array(voice_pt.numpy())
ref_s = voice[len(input_ids_list)]
print(f"\nvoice pack shape: {voice.shape}")
print(f"ref_s shape (indexed by phoneme count {len(input_ids_list)}): {ref_s.shape}")
print(f"ref_s[0,:5]: {ref_s[0, :5].tolist()}")

In [ ]:
# BERT + duration prediction
attention_mask = (~text_mask).astype(mx.int32)
bert_dur, _ = model.bert(input_ids, attention_mask=attention_mask)
d_en = model.bert_encoder(bert_dur).transpose(0, 2, 1)

s = ref_s[:, 128:]  # prosody style

print(f"bert_dur shape: {bert_dur.shape}")
print(f"d_en shape: {d_en.shape}")
print(f"s shape: {s.shape}")
print(
    f"d_en mean: {float(mx.mean(d_en)):.6f}, std: {float(mx.sqrt(mx.mean(d_en * d_en))):.6f}"
)

# Duration encoding
d = model.predictor.text_encoder(d_en, s, input_lengths, text_mask)
print(f"\nd (duration encoded) shape: {d.shape}")
print(f"d mean: {float(mx.mean(d)):.6f}")

# Duration LSTM
x, _ = model.predictor.lstm(d)
print(f"lstm out shape: {x.shape}")

# Duration projection
duration = model.predictor.duration_proj(x)
duration = mx.sigmoid(duration).sum(axis=-1) / 1.0  # speed=1.0
pred_dur = mx.clip(mx.round(duration), a_min=1, a_max=None).astype(mx.int32)[0]
mx.eval(pred_dur)

print(f"\nduration (pre-round): {duration[0].tolist()}")
print(f"pred_dur: {pred_dur.tolist()}")
print(f"total_frames: {int(mx.sum(pred_dur))}")

In [ ]:
# Get Rust intermediate values - add a debug flag that prints durations
# Let's modify the CLI temporarily to dump duration info

# Actually, easier: write a tiny Rust test that prints the intermediate values
# For now, let's just check the Rust output duration to compare

rust_wav, rust_ph = rust_generate("Hello world")
rust_audio, _ = sf.read(rust_wav)

# From Python: total_frames=61, with hop_size=300 (upsample 10*6*5=300)
# expected samples = 61 * 300 = 18300 (plus padding)
# Python produces 36600 samples
# Let's check what the Rust side looks like

# Actually the istft hop is gen_istft_hop_size=5, and upsample is 10*6=60
# So effective hop = 60 * 5 = 300 samples per frame
# 61 frames * 300 = 18300

print(f"Python: 61 frames * 300 hop = 18300 expected samples")
print(f"Python actual: 36600 samples (2x expected - likely due to center padding)")
print(f"Rust actual: {len(rust_audio)} samples")
print(f"Rust / Python ratio: {len(rust_audio) / 36600:.2f}")

# If Rust is 60000 samples and Python is 36600, and Python has 61 frames...
# Rust would have about 61 * (60000/36600) = ~100 frames
# That suggests durations are ~1.64x too long
print(f"\nIf proportional, Rust frames ~ {61 * len(rust_audio) / 36600:.0f}")
os.unlink(rust_wav)

In [ ]:
# Compare intermediate values between Python and Rust

print("=== d_en (BERT encoder output) ===")
print(f"  Python: shape (1,512,14), mean: -0.016284, rms: 1.045752")
print(f"  Rust:   shape [1,512,14], mean:  0.000983, rms: 1.073082")
print(f"  -> Mean is different sign! RMS similar. BERT output diverges slightly.")

print("\n=== ref_s (voice embedding) ===")
print(f"  Python first5: [-0.2335, 0.1960, -0.0032, -0.1249, -0.2354]")
print(f"  Rust   first5: [-0.2342, 0.1981, -0.0026, -0.1251, -0.2329]")
print(f"  -> Very close! Voice loading is correct.")

print("\n=== s (prosody style, ref_s[:, 128:]) ===")
print(f"  Python: (checked via ref_s[0,128:133])")

# Actually let me get the Python s values
s_vals = ref_s[0, 128:133].tolist()
print(f"  Python first5: {s_vals}")
print(f"  Rust   first5: [-0.0819, 0.3838, 0.8232, 0.2056, -0.0507]")
print(f"  -> Very close! Style extraction correct.")

print("\n=== d (duration encoder output) ===")
print(f"  Python: shape (1,14,640), mean: -0.006935")
print(f"  Rust:   shape [1,14,640], mean: -0.002426")
print(f"  -> Starting to diverge (different d_en inputs -> different d)")

print("\n=== duration (pre-round) ===")
py_dur = [
    17.03,
    2.01,
    2.02,
    2.00,
    2.01,
    2.07,
    2.02,
    1.42,
    2.00,
    2.46,
    4.14,
    3.20,
    11.70,
    7.93,
]
rust_dur = [
    17.66,
    3.48,
    3.06,
    2.27,
    3.25,
    8.04,
    17.69,
    2.30,
    3.39,
    14.92,
    3.43,
    2.17,
    6.06,
    13.86,
]

print(f"  Python: {py_dur}")
print(f"  Rust:   {rust_dur}")
print(f"  Ratios: {[f'{r / p:.2f}' for p, r in zip(py_dur, rust_dur)]}")
print(f"  -> WILDLY different! Esp: space(2->18), ɜ(2.5->15), d(12->6), EOS(8->14)")
print(
    f"  -> The BERT d_en difference cascades through duration encoder -> LSTM -> projection"
)

print("\n=== pred_dur ===")
py_pred = [17, 2, 2, 2, 2, 2, 2, 1, 2, 2, 4, 3, 12, 8]
rust_pred = [18, 3, 3, 2, 3, 8, 18, 2, 3, 15, 3, 2, 6, 14]
print(f"  Python: {py_pred} -> total {sum(py_pred)}")
print(f"  Rust:   {rust_pred} -> total {sum(rust_pred)}")

print("\n=== ROOT CAUSE ===")
print("The BERT (ALBERT) encoder produces different output in Rust vs Python.")
print(
    "Small differences in d_en get amplified by the duration encoder + LSTM pipeline."
)
print("Need to investigate the CustomAlbert implementation.")

In [ ]:
# Check weight keys using the local cache
from safetensors.numpy import load_file
import os

SNAPSHOT = os.path.expanduser(
    "~/.cache/huggingface/hub/models--prince-canuma--Kokoro-82M/snapshots/e02c9eada7ce7416798af36b190a8a2dd2ecd566"
)
weights = load_file(os.path.join(SNAPSHOT, "kokoro-v1_0.safetensors"))

print("=== predictor.text_encoder weights ===")
for k in sorted(weights.keys()):
    if "predictor.text_encoder" in k:
        print(f"  {k}: {weights[k].shape}")

print("\n=== predictor.lstm weights ===")
for k in sorted(weights.keys()):
    if k.startswith("predictor.lstm"):
        print(f"  {k}: {weights[k].shape}")

print("\n=== predictor.duration_proj ===")
for k in sorted(weights.keys()):
    if "duration_proj" in k:
        print(f"  {k}: {weights[k].shape}")

In [ ]:
# Instrument Python DurationEncoder step by step
import mlx.core as mx

# Re-use the already-computed values from earlier cells
# d_en, s, input_lengths, text_mask are all still in scope

# Manually run DurationEncoder
dur_enc = model.predictor.text_encoder

x = d_en  # (1, 512, 14)
style = s  # (1, 128)
m = text_mask  # (1, 14)

x = x.transpose(2, 0, 1)  # (14, 1, 512)
s_broad = mx.broadcast_to(style, (x.shape[0], x.shape[1], style.shape[-1]))
x = mx.concatenate([x, s_broad], axis=-1)  # (14, 1, 640)
x = mx.where(m[..., None].transpose(1, 0, 2), 0.0, x)
x = x.transpose(1, 2, 0)  # (1, 640, 14)

print(f"Initial: shape={x.shape}, mean={float(mx.mean(x)):.6f}")

for bi, block in enumerate(dur_enc.lstms):
    mx.eval(x)
    print(f"  Block {bi} input: shape={x.shape}, mean={float(mx.mean(x)):.6f}")

    if hasattr(block, "fc"):  # AdaLayerNorm
        x = block(x.transpose(0, 2, 1), style).transpose(0, 2, 1)
        x = mx.concatenate([x, s_broad.transpose(1, 2, 0)], axis=1)
        x = mx.where(m[..., None].transpose(0, 2, 1), 0.0, x)
    else:  # LSTM
        x_for_lstm = x.transpose(0, 2, 1)[0]  # (14, 640)
        x_lstm_out, _ = block(x_for_lstm)
        x = x_lstm_out.transpose(0, 2, 1)
        x_pad = mx.zeros([x.shape[0], x.shape[1], m.shape[-1]])
        x_pad[:, :, : x.shape[-1]] = x
        x = x_pad

mx.eval(x)
x_final = x.transpose(0, 2, 1)
print(f"\nFinal: shape={x_final.shape}, mean={float(mx.mean(x_final)):.6f}")

In [ ]:
# Check the actual weight values for the first DurationEncoder LSTM
lstm0 = dur_enc.lstms[0]
print(f"Type: {type(lstm0)}")
print(f"Wx_forward shape: {lstm0.Wx_forward.shape}")
print(f"Wh_forward shape: {lstm0.Wh_forward.shape}")
print(
    f"bias_ih_forward: {lstm0.bias_ih_forward.shape if lstm0.bias_ih_forward is not None else None}"
)
print(
    f"bias_hh_forward: {lstm0.bias_hh_forward.shape if lstm0.bias_hh_forward is not None else None}"
)

# Get first few values of Wx_forward
wx_fwd = lstm0.Wx_forward
mx.eval(wx_fwd)
print(f"\nWx_forward[0,:5]: {wx_fwd[0, :5].tolist()}")

# Combined bias
bias_combined = lstm0.bias_ih_forward + lstm0.bias_hh_forward
mx.eval(bias_combined)
print(f"Combined bias[:5]: {bias_combined[:5].tolist()}")

In [ ]:
# Trace exact shapes through the Python DurationEncoder
x = d_en  # (1, 512, 14)
style_in = s
m = text_mask

x = x.transpose(2, 0, 1)  # (14, 1, 512)
s_broad = mx.broadcast_to(style_in, (x.shape[0], x.shape[1], style_in.shape[-1]))
x = mx.concatenate([x, s_broad], axis=-1)  # (14, 1, 640)
x = mx.where(m[..., None].transpose(1, 0, 2), 0.0, x)
x = x.transpose(1, 2, 0)  # (1, 640, 14)
print(f"After init: {x.shape}")

for bi, block in enumerate(dur_enc.lstms):
    if hasattr(block, "fc"):  # AdaLayerNorm
        xt = x.transpose(0, 2, 1)
        print(f"  Block {bi} (Norm): input shape {xt.shape}")
        x = block(xt, style_in).transpose(0, 2, 1)
        print(f"    after norm: {x.shape}")
        x = mx.concatenate([x, s_broad.transpose(1, 2, 0)], axis=1)
        print(f"    after concat: {x.shape}")
        x = mx.where(m[..., None].transpose(0, 2, 1), 0.0, x)
    else:  # LSTM
        xt = x.transpose(0, 2, 1)
        print(f"  Block {bi} (LSTM): pre-transpose shape {xt.shape}")
        xt0 = xt[0]
        print(f"    after [0]: {xt0.shape}")
        h, _ = block(xt0)
        print(f"    LSTM output: {h.shape}")
        x = h.transpose(0, 2, 1) if h.ndim == 3 else h.T
        print(f"    after transpose: {x.shape}")
        x_pad = mx.zeros([x.shape[0], x.shape[1], m.shape[-1]])
        x_pad = x_pad.at[:, :, : x.shape[-1]].add(x)
        x = x_pad
        print(f"    after pad: {x.shape}")

print(f"\nFinal: {x.transpose(0, 2, 1).shape}")

In [ ]:
# Check backward weights
lstm0 = dur_enc.lstms[0]
wx_bwd = lstm0.Wx_backward
mx.eval(wx_bwd)
print(f"Wx_backward[0,:5]: {wx_bwd[0, :5].tolist()}")

bias_bwd = lstm0.bias_ih_backward + lstm0.bias_hh_backward
mx.eval(bias_bwd)
print(f"Combined backward bias[:5]: {bias_bwd[:5].tolist()}")

wh_fwd = lstm0.Wh_forward
mx.eval(wh_fwd)
print(f"Wh_forward[0,:5]: {wh_fwd[0, :5].tolist()}")

wh_bwd = lstm0.Wh_backward
mx.eval(wh_bwd)
print(f"Wh_backward[0,:5]: {wh_bwd[0, :5].tolist()}")

In [ ]:
# Quick A/B test with the LSTM recurrence fix
import subprocess, os, tempfile
import soundfile as sf

# Use the installed binary
VOICERS_CLI = "voice"


def rust_gen(text, voice="af_heart"):
    wav = tempfile.mktemp(suffix=".wav")
    subprocess.run(
        [VOICERS_CLI, "-v", voice, "-o", wav, text],
        capture_output=True,
        text=True,
        timeout=120,
    )
    return wav


test_phrases = [
    "Hello world",
    "Good morning",
    "The quick brown fox",
    "How are you doing today?",
    "She sells seashells by the seashore.",
    "I read a book yesterday.",
    "The dog is running in the park.",
]

print(f"{'Input':<45} {'Python Whisper':<40} {'Rust Whisper'}")
print("=" * 125)

for phrase in test_phrases:
    py_wav = python_generate(phrase)
    py_heard = transcribe(py_wav) if py_wav else "FAILED"
    os.unlink(py_wav) if py_wav else None

    rust_wav = rust_gen(phrase)
    rust_heard = transcribe(rust_wav)
    os.unlink(rust_wav)

    print(f"{phrase:<45} {py_heard[:38]:<40} {rust_heard[:38]}")

print("\nDone!")

In [ ]:
# The big test: try the original phrase that sounded terrible
test_text = "Now the spaCy tokenizer uses uv run with spacy. Zero setup needed beyond having uv installed. And we get proper POS tagging."

rust_wav = rust_gen(test_text)
rust_heard = transcribe(rust_wav)
print(f"Input:   {test_text}")
print(f"Whisper: {rust_heard}")
os.unlink(rust_wav)

# Also try some harder phrases
harder = [
    "The temperature in San Francisco is sixty-five degrees Fahrenheit.",
    "She had been working on the project for three months before it was finally approved.",
    "Can you believe that the new restaurant on Fifth Avenue already has a two-hour wait?",
    "To be or not to be, that is the question.",
    "The quick brown fox jumps over the lazy dog, and then runs around the park twice.",
]

print("\n=== Harder phrases ===")
for phrase in harder:
    wav = rust_gen(phrase)
    heard = transcribe(wav)
    match = (
        "OK"
        if phrase.lower().rstrip(".,!?") in heard.lower().rstrip(".,!?")
        else "DIFF"
    )
    print(f"  [{match}] {phrase[:60]}")
    if match == "DIFF":
        print(f"       -> {heard[:80]}")
    os.unlink(wav)